# Práctica 1 — Perceptrón Multicapa (MLP) desde Cero
**Materia:** Redes Neuronales  
**Maestría en Ciencias en Inteligencia Artificial — UAQ**

## Objetivo
Implementar un perceptrón multicapa sin librerías de alto nivel, aplicando propagación hacia adelante y retropropagación manualmente sobre el dataset XOR y luego sobre Iris.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
np.random.seed(42)

## 1. Funciones de activación y sus derivadas

In [ ]:
def sigmoid(z):    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def sigmoid_d(z):  return sigmoid(z) * (1 - sigmoid(z))
def relu(z):       return np.maximum(0, z)
def relu_d(z):     return (z > 0).astype(float)
def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

z = np.linspace(-5, 5, 200)
plt.figure(figsize=(10,3))
plt.subplot(1,2,1); plt.plot(z, sigmoid(z)); plt.title('Sigmoid'); plt.grid(True, alpha=0.3)
plt.subplot(1,2,2); plt.plot(z, relu(z)); plt.title('ReLU'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('activations.png', dpi=100); plt.show()

## 2. Clase MLP con retropropagación

In [ ]:
class MLP:
    def __init__(self, layers, lr=0.01):
        self.lr = lr
        self.W, self.b = [], []
        for i in range(len(layers)-1):
            self.W.append(np.random.randn(layers[i], layers[i+1]) * np.sqrt(2/layers[i]))
            self.b.append(np.zeros((1, layers[i+1])))

    def forward(self, X):
        self.A = [X]
        self.Z = []
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = self.A[-1] @ W + b
            self.Z.append(z)
            if i < len(self.W)-1:
                self.A.append(relu(z))
            else:
                self.A.append(softmax(z))
        return self.A[-1]

    def backward(self, y_true):
        m = y_true.shape[0]
        dA = (self.A[-1] - y_true) / m
        for i in reversed(range(len(self.W))):
            dW = self.A[i].T @ dA
            db = dA.sum(axis=0, keepdims=True)
            if i > 0:
                dA = (dA @ self.W[i].T) * relu_d(self.Z[i-1])
            self.W[i] -= self.lr * dW
            self.b[i] -= self.lr * db

    def fit(self, X, y, epochs=500):
        losses = []
        for _ in range(epochs):
            yhat = self.forward(X)
            loss = -np.mean(y * np.log(yhat + 1e-9))
            losses.append(loss)
            self.backward(y)
        return losses

    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)

## 3. Aplicación al dataset Iris

In [ ]:
iris = load_iris()
X, y_raw = iris.data, iris.target.reshape(-1,1)
enc = OneHotEncoder(sparse_output=False)
y = enc.fit_transform(y_raw)
sc = StandardScaler()
X = sc.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlp = MLP(layers=[4, 16, 8, 3], lr=0.05)
losses = mlp.fit(X_train, y_train, epochs=800)

plt.figure(figsize=(8,4))
plt.plot(losses); plt.xlabel('Época'); plt.ylabel('Pérdida (cross-entropy)')
plt.title('Curva de aprendizaje MLP — Iris'); plt.grid(True, alpha=0.3)
plt.savefig('mlp_loss.png', dpi=120); plt.show()

In [ ]:
acc = (mlp.predict(X_test) == np.argmax(y_test, axis=1)).mean()
print(f"Precisión en prueba: {acc*100:.2f}%")

## Conclusiones
- La retropropagación ajusta los pesos mediante la regla de la cadena aplicada capa por capa.
- ReLU como función de activación en capas ocultas mitiga el problema del gradiente que se desvanece.
- Una arquitectura de 3 capas ocultas es suficiente para clasificar Iris con alta precisión.